In [ ]:
# ==========================================
# 1. SETUP GOOGLE COLAB & MOUNT DRIVE
# ==========================================
from google.colab import drive
import os
drive.mount('/content/drive')

# 1.2. Define your dataset path (Update this to your specific Drive folder)
directory = '/content/drive/My Drive/Kaggle/IQ-OTH-NCCD/IQ-OTHNCCD-dataset/'

categories = ['Bengin cases', 'Malignant cases', 'Normal cases']

In [ ]:
# ==========================================
# 2. SETUP & IMPORTS
# ==========================================
import os
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import ResNet50
import tensorflow.keras.backend as K

# Avoid OOM errors
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
# ==========================================
# 3. DATA LOADING & PREPROCESSING
# ==========================================

img_size = 224 # Optimized for ResNet50 input requirements [cite: 491]

def load_data():
    data = []
    for i in categories:
        path = os.path.join(directory, i)
        class_num = categories.index(i)
        for file in os.listdir(path):
            try:
                filepath = os.path.join(path, file)
                # Load as BGR and convert to RGB for Transfer Learning compatibility
                img = cv2.imread(filepath)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (img_size, img_size))
                data.append([img, class_num])
            except: pass
    random.shuffle(data)
    X = np.array([item[0] for item in data]) / 255.0 # Normalization
    y = np.array([item[1] for item in data])
    return X, y

X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Handle Class Imbalance via Cost-Sensitive Learning
weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(weights))

In [ ]:
# ==========================================
# 4. ARCHITECTURAL INNOVATION (MODIFIED HEAD)
# ==========================================
def build_clinical_resnet(input_shape=(224, 224, 3), num_classes=3):
    # Using ImageNet weights for Transfer Learning
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze bottom layers to prevent overfitting on small 996-image dataset
    for layer in base_model.layers[:140]:
        layer.trainable = False

    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(), # Reduces parameter count
        layers.BatchNormalization(),
        layers.Dropout(0.5), # Regularization for clinical reliability
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax') # Multi-class output
    ])
    return model

model = build_clinical_resnet()
model.compile(optimizer=optimizers.Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# ==========================================
# 5. SCIENTIFIC TRAINING PROTOCOL
# ==========================================
# Justifying epoch count with Early Stopping and LR reduction
train_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=4)
]

history = model.fit(X_train, y_train, epochs=50, validation_split=0.1,
                    class_weight=class_weights_dict, callbacks=train_callbacks, batch_size=16)

# **Confusion Matrix ResNet50**

In [ ]:
# ==========================================
# 6. CONFUSSION MATRIX
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Generate predictions for the ResNet50 model
# We use the test data to verify clinical accuracy
print("Generating predictions for ResNet50...")
resnet_probs = model.predict(X_test, verbose=1)
resnet_preds = np.argmax(resnet_probs, axis=1)

# Compute the Confusion Matrix
cm_resnet = confusion_matrix(y_test, resnet_preds)

# Create the Visualization
plt.figure(figsize=(10, 8))
sns.heatmap(cm_resnet, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories)

plt.title('Confusion Matrix: ResNet50 Lung Cancer Classification')
plt.ylabel('Actual Clinical Diagnosis')
plt.xlabel('AI Predicted Diagnosis')
plt.tight_layout()
plt.show()

# Print the Statistical Report for your Results Section
print("\n--- ResNet50 Performance Metrics ---")
print(classification_report(y_test, resnet_preds, target_names=categories))

# **ROC and Precision-Recall (PR) curves: ResNet50**

In [ ]:
# ==========================================
# 7. ROC VS PRECISION
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

# Generate predictions for the ResNet50 model
print("Generating predictions for ResNet50...")
resnet_probs = model.predict(X_test, verbose=1)

# Binarize labels for multi-class evaluation
# Classes: 0: Benign, 1: Malignant, 2: Normal
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
categories = ['Benign cases', 'Malignant cases', 'Normal cases']

# Setup Plotting Environment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Generate Curves for each class to show model performance
colors = ['blue', 'red', 'darkorange']

for i, color in enumerate(colors):
    # --- ROC Curve Calculation ---
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], resnet_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=color, lw=2,
             label=f'{categories[i]} (AUC = {roc_auc:.4f})')

    # --- Precision-Recall Curve Calculation ---
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], resnet_probs[:, i])
    ap = average_precision_score(y_test_bin[:, i], resnet_probs[:, i])
    ax2.plot(recall, precision, color=color, lw=2,
             label=f'{categories[i]} (AP = {ap:.4f})')

# 5. Format ROC Plot
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve: ResNet50 Diagnostic Accuracy')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

# 6. Format Precision-Recall Plot
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve: ResNet50 Performance')
ax2.legend(loc="lower left")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# **ROC Curves and Decision Curve Analysis (DCA): ResNet50**

In [ ]:
# ==========================================
# 8. DCA VS ROC
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Generate predictions for the ResNet50 model
print("Generating predictions for ResNet50...")
resnet_probs = model.predict(X_test, verbose=1)

# Binarize labels for evaluation (Focusing on Class 1: Malignant)
# Classes in your dataset: 0: Benign, 1: Malignant, 2: Normal
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_true_mal = y_test_bin[:, 1]
y_prob_mal = resnet_probs[:, 1]

# Setup Plotting Environment
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# --- PLOT 1: ROC CURVE ---
fpr, tpr, _ = roc_curve(y_true_mal, y_prob_mal)
roc_auc = auc(fpr, tpr)
ax1.plot(fpr, tpr, color='blue', lw=2, label=f'ResNet50 Malignant (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.set_xlabel('False Positive Rate (1 - Specificity)')
ax1.set_ylabel('True Positive Rate (Sensitivity)')
ax1.set_title('ROC Curve: ResNet50 Diagnostic Accuracy')
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

# --- PLOT 2: DECISION CURVE ANALYSIS (DCA) ---
def calculate_net_benefit(y_true, y_prob, thresh):
    y_pred = (y_prob >= thresh)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    n = len(y_true)
    if n == 0 or thresh == 1: return 0
    return (tp / n) - (fp / n) * (thresh / (1 - thresh))

thresholds = np.linspace(0.01, 0.99, 100)
net_benefit_res = [calculate_net_benefit(y_true_mal, y_prob_mal, t) for t in thresholds]

# Baselines for comparison
prevalence = np.mean(y_true_mal)
net_benefit_all = [prevalence - (1 - prevalence) * (t / (1 - t)) for t in thresholds]

ax2.plot(thresholds, net_benefit_res, color='blue', lw=2, label='ResNet50')
ax2.plot(thresholds, net_benefit_all, 'k--', alpha=0.3, label='Treat All (Baseline)')
ax2.axhline(y=0, color='black', lw=1, label='Treat None')

ax2.set_ylim([-0.05, prevalence + 0.1])
ax2.set_xlabel('Threshold Probability')
ax2.set_ylabel('Net Benefit')
ax2.set_title('Decision Curve Analysis (DCA): Clinical Utility')
ax2.legend(loc="upper right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# **Training and validation history: accuracy vs loss**

In [ ]:
# ==========================================
# 9. TRAINING VS VALIDATION
# ==========================================
import matplotlib.pyplot as plt

# 1. Access the history from your ResNet50 training variable
def plot_resnet_history(resnet_history_obj):
    plt.style.use('ggplot')

    # Create a figure with two subplots (side-by-side)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # --- ACCURACY HISTORY ---
    ax1.plot(resnet_history_obj.history['accuracy'], label='Training Accuracy', color='blue', lw=2)
    ax1.plot(resnet_history_obj.history['val_accuracy'], label='Validation Accuracy', color='darkblue', linestyle='--', lw=2)
    ax1.set_title('ResNet50: Training & Validation Accuracy', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)

    # --- LOSS HISTORY ---
    ax2.plot(resnet_history_obj.history['loss'], label='Training Loss', color='red', lw=2)
    ax2.plot(resnet_history_obj.history['val_loss'], label='Validation Loss', color='darkred', linestyle='--', lw=2)
    ax2.set_title('ResNet50: Training & Validation Loss', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epochs', fontsize=12)
    ax2.set_ylabel('Loss Value', fontsize=12)
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# 2. RUN THE PLOT
plot_resnet_history(history)

In [ ]:
# ==========================================
# 10. XAI INTEGRATION (GRAD-CAM)
# ==========================================

import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

def generate_gradcam(img_array, model, last_conv_layer_name):
    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

    # Get the ResNet50 base model layer from the sequential model
    resnet50_base = model.get_layer('resnet50')

    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[resnet50_base.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        last_conv_layer_output_val, preds_val = grad_model([img_tensor])

        predicted_class_idx = tf.argmax(preds_val[0])
        class_channel = preds_val[:, predicted_class_idx]

    grads = tape.gradient(class_channel, last_conv_layer_output_val)

    if grads is None:
        raise ValueError(
            "Gradients are None. This typically indicates that the target "
            "(`class_channel`) is not differentiable with respect to the source "
            "(`last_conv_layer_output_val`) within the traced computation. "
            "Ensure all operations are differentiable and `tape.watch()` is correctly applied."
        )

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output_val = last_conv_layer_output_val[0]
    heatmap = last_conv_layer_output_val @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    max_heatmap_val = tf.math.reduce_max(heatmap)
    if max_heatmap_val == 0:
        heatmap = tf.zeros_like(heatmap, dtype=tf.float32)
    else:
        heatmap = tf.maximum(heatmap, 0) / (max_heatmap_val + tf.keras.backend.epsilon())
    return heatmap.numpy()

# Visualizing results
idx = 0 # First test image
sample_img = np.expand_dims(X_test[idx], axis=0) # This is a NumPy array
heatmap = generate_gradcam(img_array=sample_img, model=model, last_conv_layer_name='conv5_block3_out')

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(X_test[idx]); plt.title('Original CT')
plt.subplot(1, 2, 2); plt.imshow(X_test[idx]); plt.imshow(cv2.resize(heatmap, (224, 224)), alpha=0.4, cmap='jet'); plt.title('Grad-CAM XAI')
plt.show()

In [ ]:
# ==========================================
# 11. CLINICAL EVALUATION METRICS
# ==========================================
y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=categories))